In [1]:
# Level 3 : Data Preparation and Transformation

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("TrainSchedulePipeline") \
.master("local[*]") \
.getOrCreate()

In [3]:
import pandas as pd

df_pandas = pd.read_csv("/home/jovyan/work/cleaned_data.csv")
print(df_pandas.head())

   SN  Train_No Station_Code   1A   2A   3A   SL  Station_Name  Route_Number  \
0   1       107          SWV  100  100  100  100  SAWANTWADI R             1   
1   2       107         THVM  260  228  196  164        THIVIM             1   
2   3       107         KRMI  345  296  247  198       KARMALI             1   
3   4       107          MAO  490  412  334  256   MADGOAN JN.             1   
4   1       108          MAO  100  100  100  100   MADGOAN JN.             1   

  Arrival_time Departure_Time  Distance  
0     00:00:00       10:25:00         0  
1     11:06:00       11:08:00        32  
2     11:28:00       11:30:00        49  
3     12:10:00       00:00:00        78  
4     00:00:00       20:30:00         0  


In [4]:
df = spark.read.csv("/home/jovyan/work/cleaned_data.csv",
                    header = True,
                    inferSchema = True
                   )
df.show(5)

+---+--------+------------+---+---+---+---+------------+------------+-------------------+-------------------+--------+
| SN|Train_No|Station_Code| 1A| 2A| 3A| SL|Station_Name|Route_Number|       Arrival_time|     Departure_Time|Distance|
+---+--------+------------+---+---+---+---+------------+------------+-------------------+-------------------+--------+
|  1|     107|         SWV|100|100|100|100|SAWANTWADI R|           1|2026-05-21 00:00:00|2026-05-21 10:25:00|       0|
|  2|     107|        THVM|260|228|196|164|      THIVIM|           1|2026-05-21 11:06:00|2026-05-21 11:08:00|      32|
|  3|     107|        KRMI|345|296|247|198|     KARMALI|           1|2026-05-21 11:28:00|2026-05-21 11:30:00|      49|
|  4|     107|         MAO|490|412|334|256| MADGOAN JN.|           1|2026-05-21 12:10:00|2026-05-21 00:00:00|      78|
|  1|     108|         MAO|100|100|100|100| MADGOAN JN.|           1|2026-05-21 00:00:00|2026-05-21 20:30:00|       0|
+---+--------+------------+---+---+---+---+-----

In [5]:
# Task 3.1 - Convert time-based fields into datetime formats

df_pandas['Arrival_time'] = pd.to_datetime(
    df_pandas['Arrival_time'],
    format = 'mixed'
)

df_pandas['Departure_Time'] = pd.to_datetime(
    df_pandas['Departure_Time'],
    format = 'mixed'
)

print("Time fields converted to Datetime format")
print("\nArrival Time dtype: ",df_pandas['Arrival_time'].dtype)
print("\nDeparture Time dtype :",df_pandas['Departure_Time'].dtype)

Time fields converted to Datetime format

Arrival Time dtype:  datetime64[ns]

Departure Time dtype : datetime64[ns]


In [6]:
# Task 3.2 - Derive journey duration per train

journey_duration = df_pandas.groupby('Train_No').agg(
    journey_start=('Departure_Time', 'first'),
    journey_end=('Arrival_time', 'last')
).reset_index()

journey_duration['journey_duration_min'] = (
    journey_duration['journey_end'] - 
    journey_duration['journey_start']
).dt.total_seconds() / 60

journey_duration['journey_duration_min'] = \
    journey_duration['journey_duration_min'].abs()

print("Journey Duration Calculated")
print("\nJourney Duration per Train:")
print(journey_duration[['Train_No',
                         'journey_start',
                         'journey_end',
                         'journey_duration_min']])

Journey Duration Calculated

Journey Duration per Train:
       Train_No       journey_start         journey_end  journey_duration_min
0           107 2026-05-21 10:25:00 2026-05-21 12:10:00                 105.0
1           108 2026-05-21 20:30:00 2026-05-21 22:25:00                 115.0
2           128 2026-05-21 19:40:00 2026-05-21 17:45:00                 115.0
3           290 2026-05-21 18:30:00 2026-05-21 02:30:00                 960.0
4           401 2026-05-21 21:30:00 2026-05-21 10:00:00                 690.0
...         ...                 ...                 ...                   ...
11108     99904 2026-05-21 08:57:00 2026-05-21 09:47:00                  50.0
11109     99905 2026-05-21 09:57:00 2026-05-21 10:40:00                  43.0
11110     99906 2026-05-21 15:40:00 2026-05-21 16:30:00                  50.0
11111     99907 2026-05-21 16:38:00 2026-05-21 17:30:00                  52.0
11112     99908 2026-05-21 23:00:00 2026-05-21 23:50:00                  50.0

[11113

In [7]:
df_pandas = df_pandas.merge(
    journey_duration[['Train_No','journey_duration_min']],
    on = 'Train_No',
    how = 'left'
)
print(df_pandas.head())

   SN  Train_No Station_Code   1A   2A   3A   SL  Station_Name  Route_Number  \
0   1       107          SWV  100  100  100  100  SAWANTWADI R             1   
1   2       107         THVM  260  228  196  164        THIVIM             1   
2   3       107         KRMI  345  296  247  198       KARMALI             1   
3   4       107          MAO  490  412  334  256   MADGOAN JN.             1   
4   1       108          MAO  100  100  100  100   MADGOAN JN.             1   

         Arrival_time      Departure_Time  Distance  journey_duration_min  
0 2026-05-21 00:00:00 2026-05-21 10:25:00         0                 105.0  
1 2026-05-21 11:06:00 2026-05-21 11:08:00        32                 105.0  
2 2026-05-21 11:28:00 2026-05-21 11:30:00        49                 105.0  
3 2026-05-21 12:10:00 2026-05-21 00:00:00        78                 105.0  
4 2026-05-21 00:00:00 2026-05-21 20:30:00         0                 115.0  


In [9]:
# Task 3.3 - Organize records by train and station sequence

df_pandas = df_pandas.sort_values(
    by=['Train_No', 'SN'],
    ascending=[True, True]
)

df_pandas = df_pandas.reset_index(drop=True)

print("Records organized by Train and Station Sequence")
print("\nSample Organized Records:")
print(df_pandas[['Train_No', 'SN',
                  'Station_Name',
                  'journey_duration_min']].head(10))

Records organized by Train and Station Sequence

Sample Organized Records:
   Train_No  SN  Station_Name  journey_duration_min
0       107   1  SAWANTWADI R                 105.0
1       107   2        THIVIM                 105.0
2       107   3       KARMALI                 105.0
3       107   4   MADGOAN JN.                 105.0
4       108   1   MADGOAN JN.                 115.0
5       108   2       KARMALI                 115.0
6       108   3        THIVIM                 115.0
7       108   4  SAWANTWADI R                 115.0
8       128   1   MADGOAN JN.                 115.0
9       128   2       KARMALI                 115.0


In [10]:
# Task 3.4 - Prepare the dataset for downstream reporting

df_pandas.to_csv(
    "/home/jovyan/work/transformed_data.csv",
    index = False
)

print("Transformed Dataset Saved Successfully")
print("File: transformed_data.csv")
print("Total Records:", len(df_pandas))
print("Total Columns:", len(df_pandas.columns))
print("\nColumns:", df_pandas.columns.tolist())

Transformed Dataset Saved Successfully
File: transformed_data.csv
Total Records: 186074
Total Columns: 13

Columns: ['SN', 'Train_No', 'Station_Code', '1A', '2A', '3A', 'SL', 'Station_Name', 'Route_Number', 'Arrival_time', 'Departure_Time', 'Distance', 'journey_duration_min']
